## Live upload worker

Run the next cell after the feasibility checkpoint. It starts a temporary public Gradio API on the active Kaggle GPU. Copy the printed `gradio.live` URL into the NeuroLens website's Presenter setup. The URL works only while this notebook session remains active.

In [ ]:
# Define the real single-case inference API. It is launched by the final cell.
from pathlib import Path
import shutil
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio>=5,<7'], check=True)

import re
import stat
import uuid
import zipfile
from dataclasses import replace

import gradio as gr


def extract_uploaded_zip(archive_path, destination):
    archive_path = Path(archive_path).resolve()
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as zipped:
        members = [member for member in zipped.infolist() if not member.is_dir()]
        if not members or len(members) > 64:
            raise ValueError('The ZIP must contain one bounded BraTS case.')
        if sum(member.file_size for member in members) > int(1.5 * 1024**3):
            raise ValueError('The uncompressed case is larger than 1.5 GB.')
        for member in members:
            if stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError('Links are not allowed inside the ZIP.')
            target = (destination / member.filename).resolve()
            if destination != target and destination not in target.parents:
                raise ValueError('The ZIP contains an unsafe file path.')
            target.parent.mkdir(parents=True, exist_ok=True)
            with zipped.open(member) as source, target.open('wb') as output:
                shutil.copyfileobj(source, output)
    return destination


def analyze_uploaded_case(case_zip):
    if not case_zip:
        raise gr.Error('Choose one BraTS ZIP.')
    try:
        from neurolens.brats import MODEL_ID, collect_environment, deterministic_summary
        archive_path = Path(case_zip).resolve()
        case_id = re.sub(r'[^A-Za-z0-9._-]+', '-', archive_path.stem).strip('-._')[:80] or 'uploaded-case'
        upload_root = WORK_ROOT / 'uploads' / f'{case_id}-{uuid.uuid4().hex[:8]}'
        input_root = extract_uploaded_zip(archive_path, upload_root / 'input')
        uploaded_cases = discover_cases(input_root)
        if len(uploaded_cases) != 1:
            raise ValueError('The ZIP must contain exactly one complete T1c/T1/T2/FLAIR case.')
        uploaded_case = replace(uploaded_cases[0], case_id=case_id)
        uploaded_validation = validate_nifti_case(uploaded_case)
        uploaded_run = run_monai_inference(uploaded_case, bundle_root, upload_root, require_cuda=True)
        uploaded_modalities, uploaded_prediction, uploaded_spacing = load_nifti_case(uploaded_case, uploaded_run.mask_path)
        uploaded_metrics = calculate_region_metrics(uploaded_prediction, uploaded_spacing, uploaded_modalities['flair'] != 0)
        uploaded_overlays = render_representative_overlays(uploaded_modalities, uploaded_prediction, upload_root / 'overlays')
        uploaded_dice = None
        if uploaded_case.ground_truth:
            import nibabel as nib
            uploaded_truth = np.rint(np.asarray(nib.as_closest_canonical(nib.load(str(uploaded_case.ground_truth))).dataobj)).astype(np.uint8)
            uploaded_dice = compute_dice_scores(uploaded_prediction, uploaded_truth)
        uploaded_result_path = save_feasibility_result(
            upload_root / 'feasibility_result.json', uploaded_case, uploaded_validation, uploaded_run,
            uploaded_metrics, uploaded_overlays, uploaded_dice,
        )
        stored = json.loads(uploaded_result_path.read_text(encoding='utf-8'))
        environment = collect_environment()
        public_result = {
            'schema_version': '2.0', 'case_id': case_id, 'research_only': True,
            'validation': {key: uploaded_validation[key] for key in ('shape', 'spacing_mm', 'orientation', 'warnings')},
            'inference': {
                'model_id': MODEL_ID, 'elapsed_seconds': stored['inference']['elapsed_seconds'],
                'peak_gpu_memory_mb': stored['inference']['peak_gpu_memory_mb'],
                'model_sha256': stored['inference']['model_sha256'],
                'mask_sha256': stored['inference']['mask_sha256'], 'hardware': environment.get('gpu'),
            },
            'measurements': uploaded_metrics,
            'evaluation': {'dice': uploaded_dice} if uploaded_dice else None,
            'summary': deterministic_summary(case_id, uploaded_metrics),
        }
        regions = uploaded_metrics['regions']
        report_path = upload_root / 'research_report.txt'
        report_path.write_text(
            f"NEUROLENS LAB - AUTOMATED RESEARCH RESULT\n\nCase: {case_id}\nModel: {MODEL_ID}\n"
            f"Runtime: {uploaded_run.elapsed_seconds:.3f} seconds\n\nMEASUREMENTS\n"
            f"Whole tumor: {regions['whole_tumor']['volume_ml']:.3f} mL\n"
            f"Tumor core: {regions['tumor_core']['volume_ml']:.3f} mL\n"
            f"Enhancing tumor: {regions['enhancing_tumor']['volume_ml']:.3f} mL\n\n"
            f"{public_result['summary']}\n\nNOT FOR CLINICAL USE\n", encoding='utf-8'
        )
        return (
            public_result, str(Path(uploaded_overlays['artifacts']['comparison']).resolve()),
            str(uploaded_run.mask_path.resolve()), str(report_path.resolve()),
        )
    except Exception as error:
        raise gr.Error(str(error)) from error


with gr.Blocks(title='NeuroLens GPU Worker') as worker_app:
    gr.Markdown('# NeuroLens GPU Worker\nResearch use only. Upload one de-identified BraTS ZIP.')
    worker_input = gr.File(label='BraTS case ZIP', file_types=['.zip'], type='filepath')
    worker_button = gr.Button('Run segmentation', variant='primary')
    worker_manifest = gr.JSON(label='Structured result')
    worker_comparison = gr.Image(label='Segmentation comparison', type='filepath')
    worker_mask = gr.File(label='NIfTI segmentation mask')
    worker_report = gr.File(label='Research report')
    worker_button.click(
        analyze_uploaded_case, worker_input,
        [worker_manifest, worker_comparison, worker_mask, worker_report],
        api_name='analyze', concurrency_limit=1,
    )

worker_app = worker_app.queue(default_concurrency_limit=1, max_size=4)

# NeuroLens: zero-training BraTS feasibility run

This checkpoint downloads NVIDIA/MONAI's pretrained `brats_mri_segmentation` v0.5.4 bundle, finds one genuine four-modality BraTS case, runs inference without training, saves the NIfTI segmentation and overlay images, calculates region volumes, and records runtime/GPU memory.

**Research/education only. This is not a medical device and must not be used for diagnosis or patient care.**

Before running: select a GPU accelerator, enable Internet access, attach a de-identified BraTS dataset, and make this repository available to the notebook. On Kaggle, add both as notebook inputs. On Colab, upload or clone the repository and set `NEUROLENS_PROJECT_ROOT` if necessary.

In [ ]:
# Locate this repository and install the model bundle's declared dependencies.
PROJECT_ROOT_OVERRIDE = None  # Example: '/content/vision-language-project'
from pathlib import Path
import os
import subprocess
import sys

def find_project_root():
    configured = PROJECT_ROOT_OVERRIDE or os.environ.get('NEUROLENS_PROJECT_ROOT')
    candidates = [Path(configured)] if configured else []
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    kaggle_inputs = Path('/kaggle/input')
    if kaggle_inputs.exists():
        matches = list(kaggle_inputs.glob('**/ml/neurolens/__init__.py'))
        candidates.extend(match.parents[2] for match in matches)
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if (resolved / 'ml' / 'neurolens' / '__init__.py').exists():
            return resolved
    raise FileNotFoundError(
        'Could not find the NeuroLens repository. Upload/attach it and set '
        'NEUROLENS_PROJECT_ROOT to its folder.'
    )

PROJECT_ROOT = find_project_root()
requirements = PROJECT_ROOT / 'ml' / 'requirements-cloud.txt'
packages = [
    line.strip() for line in requirements.read_text(encoding='utf-8').splitlines()
    if line.strip() and not line.lstrip().startswith('#')
    and not line.lower().startswith(('numpy', 'scikit-learn'))
]
packages = ['monai==1.5.2' if package.startswith('monai==') else package for package in packages]
packages.extend(['fire>=0.5,<1', 'huggingface_hub>=0.23,<2'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
sys.path.insert(0, str(PROJECT_ROOT / 'ml'))
print(f'Project: {PROJECT_ROOT}')

In [ ]:
# Fail early rather than silently attempting impractical CPU inference.
import torch
import monai

assert torch.cuda.is_available(), 'No CUDA GPU found. Select a Kaggle/Colab GPU runtime.'
print({
    'gpu': torch.cuda.get_device_name(0),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'monai': monai.__version__,
})

## Select a genuine BraTS case

The discovery code supports legacy suffixes (`t1ce`, `t1`, `t2`, `flair`) and modern suffixes (`t1c`, `t1n`, `t2w`, `t2f`). It only selects a folder containing all four modalities. Set `NEUROLENS_DATA_ROOT` to narrow the search. Do not use identifiable clinical data.

In [ ]:
from neurolens.brats import CasePaths, discover_cases, modality_from_name, validate_nifti_case

DATA_ROOT_OVERRIDE = None  # Example: '/content/brats/TrainingData'
default_data_root = '/kaggle/input' if Path('/kaggle/input').exists() else '/content/data'
DATA_ROOT = Path(DATA_ROOT_OVERRIDE or os.environ.get('NEUROLENS_DATA_ROOT', default_data_root))
CASE_INDEX = int(os.environ.get('NEUROLENS_CASE_INDEX', '0'))

# Standard NIfTI layouts; ignore zero-byte placeholders in source bundles.
cases = [
    candidate for candidate in discover_cases(DATA_ROOT)
    if all(path.is_file() and path.stat().st_size > 0 for path in candidate.ordered_images())
]

# Kaggle may expand each .nii.gz modality into a directory named *.nii.
recovered = {}
for container in DATA_ROOT.rglob('*.nii'):
    if not container.is_dir():
        continue
    modality = modality_from_name(container.name)
    members = sorted(path for path in container.rglob('*.nii') if path.is_file() and path.stat().st_size > 0)
    if modality and len(members) == 1:
        recovered.setdefault(container.parent, {})[modality] = members[0]

for folder, modalities in recovered.items():
    for path in folder.iterdir():
        if path.is_file():
            modality = modality_from_name(path.name)
            if modality and path.stat().st_size > 0:
                modalities[modality] = path
    if all(modality in modalities for modality in ('t1c', 't1', 't2', 'flair')):
        cases.append(CasePaths(
            folder.name, modalities['t1c'], modalities['t1'],
            modalities['t2'], modalities['flair'], modalities.get('ground_truth')
        ))

cases = sorted(cases, key=lambda candidate: candidate.case_id)
assert cases, f'No complete four-modality BraTS cases found under {DATA_ROOT}'
assert 0 <= CASE_INDEX < len(cases), f'CASE_INDEX must be 0..{len(cases) - 1}'
case = cases[CASE_INDEX]
validation = validate_nifti_case(case)
print(f'Found {len(cases)} complete case(s); selected {case.case_id}')
print(validation)

In [ ]:
# Download the exact pretrained bundle from MONAI's official Hugging Face repository.
from huggingface_hub import snapshot_download
from neurolens.brats import run_monai_inference

cloud_root = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
WORK_ROOT = cloud_root / 'neurolens_feasibility'
BUNDLE_DIR = cloud_root / 'neurolens_bundles'
run_dir = WORK_ROOT / case.case_id

bundle_root = BUNDLE_DIR / 'brats_mri_segmentation'
model_file = bundle_root / 'models' / 'model.pt'
if not model_file.exists():
    snapshot_download(
        repo_id='MONAI/brats_mri_segmentation',
        revision='370f7f9d062745fbac445e7fe6d6616d35df04ec',
        local_dir=bundle_root,
    )
assert model_file.is_file(), f'Missing pretrained checkpoint: {model_file}'
run = run_monai_inference(case, bundle_root, run_dir, require_cuda=True)
print({
    'mask': str(run.mask_path),
    'elapsed_seconds': round(run.elapsed_seconds, 3),
    'peak_gpu_memory_mb': run.peak_gpu_memory_mb,
    'child_peak_rss_mb': run.child_peak_rss_mb,
})

In [ ]:
# Calculate deterministic measurements, render overlays, and evaluate Dice when a label is present.
import numpy as np
from neurolens.brats import (
    calculate_region_metrics, compute_dice_scores, load_nifti_case,
    render_representative_overlays, save_feasibility_result,
)

modalities, prediction, spacing = load_nifti_case(case, run.mask_path)
metrics = calculate_region_metrics(prediction, spacing, modalities['flair'] != 0)
overlays = render_representative_overlays(modalities, prediction, run_dir / 'overlays')

dice_scores = None
if case.ground_truth:
    import nibabel as nib
    truth = np.rint(np.asarray(
        nib.as_closest_canonical(nib.load(str(case.ground_truth))).dataobj
    )).astype(np.uint8)
    dice_scores = compute_dice_scores(prediction, truth)

result_path = save_feasibility_result(
    run_dir / 'feasibility_result.json', case, validation, run,
    metrics, overlays, dice_scores,
)
if metrics['regions']['whole_tumor']['voxel_count'] == 0:
    raise RuntimeError('Inference completed, but the prediction is empty; this case does not pass the checkpoint.')
print(f'Result manifest: {result_path}')

In [ ]:
# Inspect the key evidence directly in the notebook.
import json
from IPython.display import Image as DisplayImage, display

result = json.loads(result_path.read_text(encoding='utf-8'))
display(DisplayImage(filename=result['overlays']['artifacts']['comparison']))
print(result['summary'])
print(json.dumps({
    'inference': result['inference'],
    'measurements': result['measurements'],
    'evaluation': result['evaluation'],
}, indent=2))

In [ ]:
# Package the complete checkpoint for download and academic evidence.
import shutil
from IPython.display import FileLink

archive = Path(shutil.make_archive(str(WORK_ROOT / case.case_id), 'zip', run_dir))
print(f'Archive: {archive}')
display(FileLink(str(archive)))

## Checkpoint acceptance criteria

A successful run has all of the following: (1) a genuine complete BraTS case passes shape/affine validation, (2) MONAI creates a non-corrupt `*_seg.nii.gz`, (3) `feasibility_result.json` contains model/runtime/GPU provenance and volumes, (4) the comparison image renders, and (5) Dice is recorded when the attached case includes a ground-truth segmentation.

The feasibility evidence above remains the checkpoint. The final worker cell reuses the same pinned bundle to process website uploads and must stay running while the public interface is being demonstrated.

In [ ]:
# Keep this final cell running while users submit cases from the website.
worker_app.launch(server_name='0.0.0.0', share=True, show_error=True)